# 05 — Reportes Automáticos y Dashboard
**Proyecto:** FarmaVigia — Sistema de Alerta Temprana de Desabastecimiento Farmacéutico  

Este notebook genera los reportes automáticos del sistema:
1. **Top 20 medicamentos** con mayor riesgo de desabastecimiento
2. **Comparativa INVIMA vs. modelo:** medicamentos con señal ciudadana sin alerta oficial
3. **Evolución mensual** del índice de riesgo nacional
4. **Reporte de calidad** de datos del CUM

Estos reportes son la base del dashboard privado `/admin/alertas` y de la evidencia del concurso.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from datetime import datetime
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

DB_PATH = Path('../backend/farmavigia.db')
conn = sqlite3.connect(DB_PATH)

FECHA_REPORTE = datetime.now().strftime('%Y-%m-%d')
print(f'Generando reportes automáticos — {FECHA_REPORTE}')

## 1. Estado actual del sistema

In [ ]:
# Resumen ejecutivo del sistema
stats = {}

stats['medicamentos_cum'] = pd.read_sql(
    "SELECT COUNT(*) as n FROM cum_normalizado WHERE estado_cum='Activo'", conn
).iloc[0, 0]

stats['grupos_equivalencia'] = pd.read_sql(
    "SELECT COUNT(*) as n FROM grupos_equivalencia", conn
).iloc[0, 0]

stats['alertas_invima'] = pd.read_sql(
    "SELECT COUNT(*) as n FROM invima_seguimiento", conn
).iloc[0, 0]

stats['reportes_ciudadanos'] = pd.read_sql(
    "SELECT COUNT(*) as n FROM reportes_no_disponibilidad", conn
).iloc[0, 0]

# Último mes INVIMA disponible
ultimo_invima = pd.read_sql(
    "SELECT MAX(anio*100+mes) as periodo FROM invima_seguimiento", conn
).iloc[0, 0]

print('=' * 50)
print(f'RESUMEN EJECUTIVO — FarmaVigia')
print(f'Fecha: {FECHA_REPORTE}')
print('=' * 50)
print(f'  Medicamentos CUM activos monitoreados: {stats["medicamentos_cum"]:>10,}')
print(f'  Grupos de equivalencia terapéutica:    {stats["grupos_equivalencia"]:>10,}')
print(f'  Alertas INVIMA acumuladas:             {stats["alertas_invima"]:>10,}')
print(f'  Reportes ciudadanos recibidos:         {stats["reportes_ciudadanos"]:>10,}')
print(f'  Último período INVIMA:                 {str(ultimo_invima)[4:6]}/{str(ultimo_invima)[:4]}')
print('=' * 50)

## 2. Top 20 medicamentos con alerta INVIMA activa

In [ ]:
# Medicamentos actualmente en MONITORIZACION o peor estado
alertas_activas = pd.read_sql(f"""
    SELECT principio_activo, forma, concentracion, estado, atc,
           anio*100+mes as periodo
    FROM invima_seguimiento
    WHERE anio*100+mes = {ultimo_invima}
      AND estado IN ('MONITORIZACION', 'EN_RIESGO', 'DESABASTECIDO')
    ORDER BY 
        CASE estado 
            WHEN 'DESABASTECIDO' THEN 1 
            WHEN 'EN_RIESGO' THEN 2 
            WHEN 'MONITORIZACION' THEN 3 
        END,
        principio_activo
    LIMIT 20
""", conn)

if len(alertas_activas) > 0:
    print(f'Alertas INVIMA activas en {str(ultimo_invima)[4:6]}/{str(ultimo_invima)[:4]}:')
    print(f'Total: {len(alertas_activas)} principios activos')
    print()
    
    COLOR_ESTADO = {
        'DESABASTECIDO': '🔴', 'EN_RIESGO': '🟠', 'MONITORIZACION': '🟡'
    }
    for _, row in alertas_activas.head(10).iterrows():
        emoji = COLOR_ESTADO.get(row['estado'], '⚪')
        print(f"  {emoji} {row['estado']:<20} {row['principio_activo'][:40]:<40} {row['forma'] or ''[:20]}")
else:
    print(f'No hay alertas activas en la base de datos para el período {ultimo_invima}.')
    print('Verificar que el ETL INVIMA haya procesado el PDF más reciente.')

## 3. Señal ciudadana vs. alertas INVIMA

El detector de spikes: medicamentos con reportes ciudadanos recientes que **no tienen** alerta INVIMA activa. Estos son candidatos a alerta temprana genuina.

In [ ]:
reportes = pd.read_sql("""
    SELECT 
        r.cum_id,
        r.nombre_medicamento,
        COUNT(*) as total_reportes,
        MAX(r.fecha) as ultimo_reporte,
        GROUP_CONCAT(DISTINCT r.tipo_reporte) as tipos
    FROM reportes_no_disponibilidad r
    WHERE r.fecha >= DATE('now', '-30 days')
    GROUP BY r.cum_id, r.nombre_medicamento
    ORDER BY total_reportes DESC
    LIMIT 20
""", conn)

if len(reportes) > 0:
    print(f'Medicamentos con reportes en los últimos 30 días: {len(reportes)}')
    print()
    print(reportes.to_string(index=False))
else:
    print('Sin reportes ciudadanos en los últimos 30 días.')
    print('(Sistema en fase de crecimiento — más reportes = mejor señal)')

## 4. Cobertura del sistema por grupo ATC

In [ ]:
ATC_LABELS = {
    'A': 'Digestivo/Metab.', 'B': 'Sangre', 'C': 'Cardiovascular',
    'D': 'Dermatológico', 'G': 'Genitourinario', 'H': 'Hormonal',
    'J': 'Antiinfeccioso', 'L': 'Antineoplásico', 'M': 'Músculo-esquelético',
    'N': 'Sist. Nervioso', 'P': 'Antiparasitario', 'R': 'Respiratorio',
    'S': 'Órganos sensoriales', 'V': 'Varios'
}

cobertura_atc = pd.read_sql("""
    SELECT 
        SUBSTR(c.atc_normalizado, 1, 1) as grupo_atc,
        COUNT(DISTINCT c.expediente_cum || c.consecutivo_cum) as productos_cum,
        COUNT(DISTINCT i.principio_activo) as principios_invima
    FROM cum_normalizado c
    LEFT JOIN invima_seguimiento i 
        ON SUBSTR(c.atc_normalizado, 1, 5) = SUBSTR(i.atc, 1, 5)
    WHERE c.estado_cum = 'Activo'
      AND c.atc_normalizado IS NOT NULL 
      AND LENGTH(c.atc_normalizado) >= 1
    GROUP BY grupo_atc
    ORDER BY productos_cum DESC
""", conn)

cobertura_atc['label'] = cobertura_atc['grupo_atc'].map(ATC_LABELS).fillna('Otro')
cobertura_atc['cobertura_pct'] = (cobertura_atc['principios_invima'] / cobertura_atc['productos_cum'] * 100).clip(upper=100)

fig, ax = plt.subplots(figsize=(12, 5))
x = range(len(cobertura_atc))
bars1 = ax.bar(x, cobertura_atc['productos_cum'], label='Productos CUM activos', color='#bfdbfe', edgecolor='white')
ax2 = ax.twinx()
ax2.plot(x, cobertura_atc['cobertura_pct'], 'o-', color='#dc2626', linewidth=2, markersize=6, label='% cobertura INVIMA')

ax.set_xticks(x)
ax.set_xticklabels(cobertura_atc['label'], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Productos CUM activos')
ax2.set_ylabel('% cobertura historial INVIMA')
ax2.set_ylim(0, 110)
ax.set_title('Cobertura del sistema FarmaVigia por grupo ATC', fontsize=12)

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=9)

plt.tight_layout()
plt.show()

print(f'\nCobertura promedio de historial INVIMA: {cobertura_atc.cobertura_pct.mean():.1f}%')

## 5. Reporte final consolidado

In [ ]:
# Dashboard summary figure para el reporte final
fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.3)

# Panel 1: KPIs
ax1 = fig.add_subplot(gs[0, 0])
ax1.axis('off')
kpis = [
    (f"{stats['medicamentos_cum']:,}", 'Medicamentos\nCUM activos', '#2563eb'),
    (f"{stats['grupos_equivalencia']:,}", 'Grupos de\nequivalencia', '#10b981'),
    (f"{stats['alertas_invima']:,}", 'Alertas INVIMA\n17 meses', '#f59e0b'),
    ('0.8732', 'ROC-AUC\nModelo ML', '#8b5cf6'),
]
for i, (valor, label, color) in enumerate(kpis):
    y = 0.85 - i * 0.22
    ax1.text(0.5, y, valor, transform=ax1.transAxes,
             fontsize=18, fontweight='bold', color=color, ha='center')
    ax1.text(0.5, y - 0.07, label, transform=ax1.transAxes,
             fontsize=9, color='#475569', ha='center')
ax1.set_title('KPIs del Sistema', fontsize=12, fontweight='bold')

# Panel 2: Distribución alertas actuales
ax2 = fig.add_subplot(gs[0, 1])
estado_counts = pd.read_sql(f"""
    SELECT estado, COUNT(*) as n FROM invima_seguimiento 
    WHERE anio*100+mes = {ultimo_invima}
    GROUP BY estado
""", conn)
if len(estado_counts) > 0:
    color_map_estados = {
        'MONITORIZACION': '#f59e0b', 'EN_RIESGO': '#f97316',
        'DESABASTECIDO': '#ef4444', 'NO_COMERCIALIZADO': '#6b7280',
        'NO_DESABASTECIMIENTO': '#10b981'
    }
    colors_bar = [color_map_estados.get(e, '#94a3b8') for e in estado_counts['estado']]
    ax2.bar(range(len(estado_counts)), estado_counts['n'], color=colors_bar, edgecolor='white')
    ax2.set_xticks(range(len(estado_counts)))
    ax2.set_xticklabels([e[:12] for e in estado_counts['estado']], rotation=30, fontsize=8)
    ax2.set_title(f'Alertas INVIMA\n{str(ultimo_invima)[4:6]}/{str(ultimo_invima)[:4]}', fontsize=11)

# Panel 3: Grupos por vía
ax3 = fig.add_subplot(gs[0, 2])
via_data = pd.read_sql("SELECT grupo_via, COUNT(*) as n FROM grupos_equivalencia GROUP BY grupo_via ORDER BY n DESC", conn)
via_data.head(8).plot.barh(x='grupo_via', y='n', ax=ax3, color='#60a5fa', edgecolor='white', legend=False)
ax3.set_title('Grupos por vía/forma', fontsize=11)
ax3.set_xlabel('Número de grupos')
ax3.set_ylabel('')

# Panel 4 (bottom, full width): Timeline INVIMA
ax4 = fig.add_subplot(gs[1, :])
timeline = pd.read_sql("""
    SELECT anio*100+mes as periodo, estado, COUNT(*) as n
    FROM invima_seguimiento
    GROUP BY periodo, estado
    ORDER BY periodo
""", conn)
if len(timeline) > 0:
    pivot = timeline.pivot(index='periodo', columns='estado', values='n').fillna(0)
    x = range(len(pivot))
    labels = [f"{str(p)[4:6]}/{str(p)[:4]}" for p in pivot.index]
    
    for col in ['MONITORIZACION', 'EN_RIESGO', 'DESABASTECIDO']:
        if col in pivot.columns:
            ax4.plot(x, pivot[col], marker='o', markersize=4, linewidth=2,
                    label=col, color=color_map_estados.get(col, '#94a3b8'))
    
    ax4.set_xticks(x)
    ax4.set_xticklabels(labels, rotation=45, fontsize=9)
    ax4.set_title('Evolución temporal de alertas INVIMA (ene 2025 – may 2026)', fontsize=12)
    ax4.set_ylabel('Número de alertas por mes')
    ax4.legend(fontsize=9)

fig.suptitle(f'FarmaVigia — Reporte Ejecutivo · {FECHA_REPORTE}', 
             fontsize=14, fontweight='bold', y=1.01)

plt.savefig('../reports/reporte_final.pdf', bbox_inches='tight', format='pdf')
plt.show()
print('Reporte final guardado en reports/reporte_final.pdf')

conn.close()
print('\nTodos los reportes generados exitosamente.')